In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import joblib
import numpy as np

from sklearn_evaluation import plot
from sklearn.metrics import r2_score

from datetime import datetime

from sklearn.ensemble import (
    AdaBoostRegressor,
    RandomForestRegressor,
    GradientBoostingRegressor,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [ ]:
MAX_DEPTH = -30.0

# Get all the csvs in the data directory
data_dir = Path("data")
regions = gpd.read_file('postcards.geojson')

# Read them and merge them into a single dataframe
all = []
atolls = []
islands = []

for region in regions.itertuples():
    csv = data_dir / f"training/{region.name}_land_mask.csv"
    is_atoll = region.type == "Atoll"

    gdf = gpd.read_file(csv)

    # Make sure everything can be converted to a float
    for col in gdf.columns:
        gdf[col] = gdf[col].astype(float)

    # Replace infinite values with NaN
    gdf = gdf.replace([float('-inf'), float('inf')], float('nan'))

    # Drop rows with missing values
    gdf = gdf.dropna()
    gdf = gdf[gdf.depth > MAX_DEPTH]

    print(f"Read {csv} ({'Atoll' if is_atoll else 'Island'}) with {len(gdf)} data points less than {MAX_DEPTH} m")

    # Get them all and put them in lists
    all.append(gdf)
    if is_atoll:
        atolls.append(gdf)
    else:
        islands.append(gdf)

all_data = gpd.GeoDataFrame(pd.concat(all, ignore_index=True))
atolls_data = gpd.GeoDataFrame(pd.concat(atolls, ignore_index=True))
islands_data = gpd.GeoDataFrame(pd.concat(islands, ignore_index=True))

print(f"\nTotal data points: {len(all_data)}, Atolls: {len(atolls_data)}, Islands: {len(islands_data)}")

# Get rid of specifc values, which I think come from inaccurate digitisation of contours
filtered = all_data[~all_data.depth.isin([0, -5, -8, -10, 12.5, -15, -17.5, -20, -25, -30])]

print(f"Filtered data points excluting round numbers: {len(filtered)}")

In [ ]:
# We use a random forest regressor to do a first pass
# on the data, and remove outliers
data = filtered

# Split the data into training and testing
train, test = train_test_split(data, test_size=0.3)

# Define the variables and the target
depth = train["depth"]
variables = train.drop(columns=["depth", "x", "y"])

# Define the model
# regressor = AdaBoostRegressor()
regressor = RandomForestRegressor()
# regressor = GradientBoostingRegressor()

# Train the model
rf_model = regressor.fit(variables, depth)

# Evaluate on our test data
test_depth = test["depth"]
test_variables = test.drop(columns=["depth", "x", "y"])

predictions = rf_model.predict(test_variables)
mse = mean_squared_error(test_depth, predictions)
mae = mean_absolute_error(test_depth, predictions)
r2 = r2_score(test_depth, predictions)

print(f"R² score: {r2:.3f}")
print(f"Mean squared error: {mse:.3f}")
print(f"Mean absolute error: {mae:.3f}")

In [ ]:
# Clean up training data
rf_preds = rf_model.predict(data.drop(columns=["depth", "x", "y"]))
residuals = np.abs(rf_preds - data["depth"])

threshold = 3.0
mask = residuals < threshold

filtered_data = data[mask]

print(f"Filtered data points: {len(filtered_data)} out of {len(data)}")

In [ ]:
# New RF model on filtered data

# Split the data into training and testing
train, test = train_test_split(filtered_data, test_size=0.3)

# Define the variables and the target
depth = train["depth"]
variables = train.drop(columns=["depth", "x", "y"])

# Define the model
# regressor = AdaBoostRegressor()
regressor = RandomForestRegressor()
# regressor = GradientBoostingRegressor()

# Train the model
rf_model = regressor.fit(variables, depth)

# Evaluate on our test data
test_depth = test["depth"]
test_variables = test.drop(columns=["depth", "x", "y"])

predictions = rf_model.predict(test_variables)

mse = mean_squared_error(test_depth, predictions)
mae = mean_absolute_error(test_depth, predictions)
r2 = r2_score(test_depth, predictions)

print(f"R² score: {r2:.3f}")
print(f"Mean squared error: {mse:.3f}")
print(f"Mean absolute error: {mae:.3f}")

In [ ]:
out = "models/2025_04_16_rf.joblib"

# Write out the model
joblib.dump(rf_model, out)

# Write out a little metadata file
with open(out.replace(".joblib", ".txt"), "w") as f:
    f.write(f"Mean squared error: {mse:0.3f}\n")
    f.write(f"Mean absolute error: {mae:0.3f}\n")
    f.write(f"R² score: {r2:0.3f}\n")
    f.write(f"Model: random forest\n")
    f.write(f"Data: all, limit at {MAX_DEPTH} m\n")
    f.write(f"Masked: land only\n")
    f.write(f"Date and time: {datetime.now()}\n")
    f.write(f"File: {out}\n")

In [ ]:
import torch
import numpy as np
import torch.nn as nn
from skorch.regressor import NeuralNetRegressor
from skorch.callbacks import EarlyStopping, EpochScoring
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from src.utils import get_regressor


# Main training function
def train_bathymetry_nn_model(filtered_data):
    # THIS DIDN'T HELP!
    # values_scaled = filtered_data.copy().astype(np.float32)
    # values_scaled["depth"] = (values_scaled["depth"]) / -30.0

    # Split train/test
    train, test = train_test_split(
        filtered_data.astype(np.float32), test_size=0.3, random_state=42
    )

    # Extract features and target
    X_train = train.drop(columns=["depth", "x", "y"])
    y_train = train["depth"]

    X_test = test.drop(columns=["depth", "x", "y"])
    y_test = test["depth"]

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model
    regressor = get_regressor(shape=X_train.shape[1])

    # Fit the model
    regressor.fit(X_train_scaled, y_train)

    # Predict and evaluate
    y_pred = regressor.predict(X_test_scaled).flatten()
    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print("\n🧠 Final Model Performance:")
    print(f"R² score: {r2:.3f}")
    print(f"Mean squared error: {mse:.3f}")
    print(f"Mean absolute error: {mae:.3f}")

    return regressor, scaler, (X_test_scaled, y_test, y_pred), (r2, mse, mae)


nn_model, scaler, (X_test_scaled, y_test, y_pred), (r2, mse, mae) = (
    train_bathymetry_nn_model(filtered_data)
)

In [ ]:
values_scaled = filtered_data.copy().astype(np.float32)
values_scaled["depth"] = (values_scaled["depth"]) / -30.0

# Split train/test
train, test = train_test_split(values_scaled, test_size=0.3, random_state=42)

# Extract features and target
X_train = train.drop(columns=["depth", "x", "y"])
y_train = train["depth"]

X_test = test.drop(columns=["depth", "x", "y"])
y_test = test["depth"]

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
train_preds = nn_model.predict(X_test_scaled).flatten()

import matplotlib.pyplot as plt
plt.scatter(y_test, train_preds, alpha=0.01)
plt.xlabel("True Depth")
plt.ylabel("Predicted Depth")
plt.title("Training Predictions vs True Depth")
plt.grid(True)
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression
from matplotlib import pyplot as plt
import numpy as np

from sklearn_evaluation.plot.regression import _set_ax_settings

# Do this plot, but with alpha on the points
# plot.residuals(test_depth, predictions)

y_true = y_test
y_pred = train_preds


_, ax = plt.subplots()

default_color = "#00B0FF"

# horizontal line for residual=0
ax.axhline(y=0, color=default_color)
ax.scatter(y_pred, y_true - y_pred, c=default_color, edgecolors=default_color, alpha=0.01)

_set_ax_settings(ax, "Predicted Value", "Residuals", "Residuals Plot")

In [ ]:
plot.feature_importances(nn_model, feature_names=variables.columns)

In [ ]:
# Do this plot, but with alpha on the points
# plot.regression.prediction_error(test_depth, predictions)

_, ax = plt.subplots()
regression = LinearRegression()

if isinstance(y_true, pd.Series):
    y_true = y_true.values
y_reshaped = y_true.reshape((-1, 1))

# it is necessary to fit the model with y_true and y_pred
# to get the best fit line representing the error trend
regression.fit(y_reshaped, y_pred)
x = np.linspace(min(min(y_true), min(y_pred)), max(max(y_true), max(y_pred)))
y = regression.intercept_ + regression.coef_ * x

default_color = "#00B0FF"

ax.plot(x, y, color="#666", label="best fit", linewidth=1)

# identity line
ax.plot(
    x, x, label="identity", color="#000", linewidth=1, alpha=0.5, linestyle="dashed"
)

# scatter plot
ax.scatter(y_true, y_pred, c=default_color, edgecolors=default_color, alpha=0.01)

# R2
r2 = regression.score(y_reshaped, y_pred)
plt.plot([], [], " ", label=f"R2 = {round(r2, 5)}")

_set_ax_settings(ax, "y_true", "y_pred", "Prediction Error")
ax.legend(loc="upper left")

In [ ]:
import torch
import os
import zipfile

base = "2025_04_16d_nn"

# Ensure working files directory
os.makedirs("models", exist_ok=True)

# Save weights
torch.save(nn_model.module_.state_dict(), f"models/{base}_weights.pt")

# Save scaler
joblib.dump(scaler, f"models/{base}_scaler.pkl")

# Write out a little metadata file
with open(f"models/{base}_info.txt", "w") as f:
    f.write(f"Mean squared error: {mse:.3f}\n")
    f.write(f"Mean absolute error: {mae:.3f}\n")
    f.write(f"R² score: {r2:.3f}\n")
    f.write(f"Model: custom nn\n")
    f.write(f"Data: all, limit at {MAX_DEPTH} m\n")
    f.write(f"Masked: land only\n")
    f.write(f"Date and time: {datetime.now()}\n")
    f.write(f"Includes scaler\n")
    f.write("Includes tweaks on advice from geoneon\n")
    f.write(f"File: {out}\n")

# Create zip bundle
with zipfile.ZipFile(out, "w") as zipf:
    zipf.write(f"models/{base}_weights.pt", arcname=f"{base}_weights.pt")
    zipf.write(f"models/{base}_scaler.pkl", arcname=f"{base}_scaler.pkl")
    zipf.write(f"models/{base}_info.txt", arcname=f"{base}_info.txt")
